In [58]:
import numpy as np
import pandas as pd 
import torch
import torch.nn as nn
import torch.optim as optim
import pickle
import os
import torchvision.models as models  
import tqdm
import sys
sys.path.append('/bohr/train-4gug/v2')
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataloader import load_data
import matplotlib.pyplot as plt



import torch
import torch.nn as nn
import torch.nn.functional as F


import torch
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.nn as nn
import torch.nn.functional as F


import torch
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(6, 32, kernel_size=3, stride=2, padding=1),  # Downsample
            nn.MaxPool2d(2),
            nn.BatchNorm2d(32),
            nn.Dropout(p=0.2),
            nn.ReLU()
        )

        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # Downsample
            nn.MaxPool2d(2),
            nn.BatchNorm2d(64),
            nn.Dropout(p=0.2),
            nn.ReLU()
        )

        self.enc3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=7, stride=2, padding=3),  # Downsample
            nn.MaxPool2d(2),
            nn.BatchNorm2d(128),
            nn.Dropout(p=0.2),
            nn.ReLU()
        )

        self.bottleneck = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=5, padding=2),
            nn.BatchNorm2d(256),
            nn.Dropout(p=0.2),
            nn.ReLU()
        )

        # Decoder
        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2),  # Upsample
            nn.BatchNorm2d(128),
            nn.Dropout(p=0.2),
            nn.ReLU()
        )

        self.dec2 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),  # Upsample
            nn.BatchNorm2d(64),
            nn.Dropout(p=0.2),
            nn.ReLU()
        )

        self.dec3 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),  # Upsample
            nn.BatchNorm2d(32),
            nn.Dropout(p=0.2),
            nn.ReLU()
        )

        self.out_conv = nn.Conv2d(32, 2, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]

        x = self.enc1(x)
        s1 = x
        x = self.enc2(x)
        s2 = x
        x = self.enc3(x)
        s3 = x
        x = self.bottleneck(x)
        x = self.dec1(x)
        x = self.dec2(x)
        x = self.dec3(x)

        x = self.out_conv(x)
        x = F.interpolate(x, size=input_size, mode='bilinear', align_corners=False)  # Exact match

        return x
def train(model, train_loader, test_loader, optimizer, criterion, num_epochs=100):
    train_losses = []
    val_losses = []
    min_val_loss = 10000000
    best_model = model
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        batch_count = 0
        
        for images, labels, _ in train_loader:
            images = images.cuda() if torch.cuda.is_available() else images
            labels = labels.cuda() if torch.cuda.is_available() else labels
            
            outputs = model(images)
            outputs = outputs.view(outputs.size(0), outputs.size(1), -1)  # [B, C, H*W]
            labels = labels.view(labels.size(0), -1)  # [B, H*W]
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            batch_count += 1
        
        avg_train_loss = epoch_loss / batch_count
        train_losses.append(avg_train_loss)
        
        model.eval()
        val_loss = 0.0
        val_batch_count = 0
        img_pred = None
        
        with torch.no_grad():
            for images, labels, _ in test_loader:
                images = images.cuda() if torch.cuda.is_available() else images
                labels = labels.cuda() if torch.cuda.is_available() else labels
                
                outputs = model(images)
                img_pred = torch.argmax(outputs, dim=1)

                outputs = outputs.view(outputs.size(0), outputs.size(1), -1)
                labels = labels.view(labels.size(0), -1)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                val_batch_count += 1
        
        
        
        avg_val_loss = val_loss / (val_batch_count + 1)

        if(avg_val_loss < min_val_loss):
            min_val_loss = avg_val_loss
            best_model = copy.deepcopy(model)
        
        val_losses.append(avg_val_loss)
        
        print(f'Epoch [{epoch+1}/{num_epochs}], '
              f'Train Loss: {avg_train_loss:.4f}, '
              f'Val Loss: {avg_val_loss:.4f}')

            # if((epoch + 1) % 10 == 0):
            #     plt.matshow(img_pred.detach().cpu().numpy()[0])
            #     plt.show()

    return best_model

data_path = '/bohr/train-4gug/v2/training_set'

train_loader, test_loader = load_data(
    base_path=data_path,
    batch_size=64,  
    test_size=0.2,
)

model = MyModel()
if torch.cuda.is_available():
    model = model.cuda()

class_weights = [1, 500]
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).cuda()
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=0.001) 

model = train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    criterion=criterion,
    num_epochs=150
)

In [4]:
# Please write your model code (including the necessary imported modules, such as torch and torch.nn) below to generate a model structure file that can be easily loaded by the grading platform
model_code = """  
import numpy as np
import pandas as pd 
import torch
import torch.nn as nn
import torch.optim as optim
import pickle
import os
import copy
import torchvision.models as models  
import tqdm
import sys
sys.path.append('/bohr/train-4gug/v2')
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataloader import load_data
import matplotlib.pyplot as plt
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(6, 32, kernel_size=3, stride=2, padding=1),  # Downsample
            nn.MaxPool2d(2),
            nn.BatchNorm2d(32),
            nn.Dropout(p=0.2),
            nn.ReLU()
        )

        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # Downsample
            nn.MaxPool2d(2),
            nn.BatchNorm2d(64),
            nn.Dropout(p=0.2),
            nn.ReLU()
        )

        self.enc3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=7, stride=2, padding=3),  # Downsample
            nn.MaxPool2d(2),
            nn.BatchNorm2d(128),
            nn.Dropout(p=0.2),
            nn.ReLU()
        )

        self.bottleneck = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=5, padding=2),
            nn.BatchNorm2d(256),
            nn.Dropout(p=0.2),
            nn.ReLU()
        )

        # Decoder
        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2),  # Upsample
            nn.BatchNorm2d(128),
            nn.Dropout(p=0.2),
            nn.ReLU()
        )

        self.dec2 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),  # Upsample
            nn.BatchNorm2d(64),
            nn.Dropout(p=0.2),
            nn.ReLU()
        )

        self.dec3 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),  # Upsample
            nn.BatchNorm2d(32),
            nn.Dropout(p=0.2),
            nn.ReLU()
        )

        self.out_conv = nn.Conv2d(32, 2, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]

        x = self.enc1(x)
        s1 = x
        x = self.enc2(x)
        s2 = x
        x = self.enc3(x)
        s3 = x
        x = self.bottleneck(x)
        x = self.dec1(x)
        x = self.dec2(x)
        x = self.dec3(x)

        x = self.out_conv(x)
        x = F.interpolate(x, size=input_size, mode='bilinear', align_corners=False)  # Exact match

        return x

"""
# Write code to file
with open('submission_model.py', 'w',encoding="utf-8") as f:
    f.write(model_code)
print("submission_model.py file has been generated.")

In [6]:
# Save the parameters of the model
torch.save(model.state_dict(), 'submission_dic.pth')
print("submission_dic.pth file has been saved.")

In [8]:
# This block mainly specifies the submission format of this question.
import zipfile
import os

# Define the files to zip and the zip file name.
files_to_zip = ['submission_model.py', 'submission_dic.pth']
zip_filename = 'submission.zip'

# Create a zip file
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        # Add the file to the zip fil
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} Created successfully!')